# Importing the dataset

In [39]:
import pandas as pd

airline_df = pd.read_csv('/content/Airline_clean.csv', keep_default_na=False)
print('Airline_clean.csv head:')
display(airline_df.head())

Airline_clean.csv head:


,Passenger ID,First Name,Last Name,Gender,Age,Nationality,Airport Name,Airport Country Code,Country Name,Airport Continent,Departure Date,Airport Code,Pilot Name,Flight Status,passenger_count,is_international,age_group
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,28/06/2022,CXF,Fransisco Hazeldine,On Time,1,1,Middle Aged (45-64)
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,26/12/2022,YCO,Marla Parsonage,On Time,1,1,Middle Aged (45-64)
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,EU,18/01/2022,GNB,Rhonda Amber,On Time,1,1,Senior (65+)
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,16/09/2022,YND,Kacie Commucci,Delayed,1,1,Senior (65+)
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,25/02/2022,SEE,Ebonee Tree,On Time,1,1,Young Adult (18-24)


In [40]:
countries_df = pd.read_csv('/content/Countries_clean.csv', keep_default_na=False)
print('Countries_clean.csv head:')
display(countries_df.head())

Countries_clean.csv head:


,id,code,name,continent
0,302672,AD,Andorra,EU
1,302618,AE,United Arab Emirates,AS
2,302619,AF,Afghanistan,AS
3,302722,AG,Antigua and Barbuda,NA
4,302723,AI,Anguilla,NA


In [41]:
airport_df = pd.read_csv('/content/Airport_clean.csv', keep_default_na=False)
print('Airport_clean.csv head:')
display(airport_df.head())

Airport_clean.csv head:


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,iata_code
0,6623,01SC,small_airport,York Airport,35.032101,-81.252800,779.0,NA,US,US-SC,York,no,THV
1,6625,01TE,small_airport,Smith Field,32.737598,-96.428001,505.0,NA,US,US-TX,Forney,no,SLG
2,322099,02MD,small_airport,Garner Field,38.672544,-76.709739,141.0,NA,US,US-MD,Brandywine,no,UVA
3,4650,03N,small_airport,Utirik Airport,11.222219,169.851429,4.0,OC,MH,MH-UTI,Utirik Island,yes,UTK
4,324414,06NV,small_airport,Silver Creek Airport,39.098333,-114.150277,5556.0,NA,US,US-NV,Baker,no,SVK


# Creating dim_location

In [42]:
import pandas as pd

dim_location = pd.DataFrame()
print("Empty DataFrame created:")
display(dim_location)

Empty DataFrame created:


""


In [43]:
dim_location = countries_df[['code', 'name', 'continent']].copy()

dim_location.columns = ['country_code', 'country_name', 'continent_code']

In [44]:
#Add continent_name by mapping the codes
CONTINENT_NAMES = {
    'AF': 'Africa',
    'AN': 'Antarctica',
    'AS': 'Asia',
    'EU': 'Europe',
    'NA': 'North America',
    'OC': 'Oceania',
    'SA': 'South America',
}
dim_location['continent_name'] = dim_location['continent_code'].map(CONTINENT_NAMES)

#Add location_key as the first column
dim_location.insert(0, 'location_key', range(1, len(dim_location) + 1))

In [45]:
dim_location.head(5)

,location_key,country_code,country_name,continent_code,continent_name
0,1,AD,Andorra,EU,Europe
1,2,AE,United Arab Emirates,AS,Asia
2,3,AF,Afghanistan,AS,Asia
3,4,AG,Antigua and Barbuda,NA,North America
4,5,AI,Anguilla,NA,North America


# Creating dim_passenger

In [46]:
dim_passenger = pd.DataFrame()
print("Empty DataFrame created:")
display(dim_passenger)

Empty DataFrame created:


""


In [47]:
dim_passenger = airline_df[['Passenger ID', 'Age', 'age_group', 'Gender', 'Nationality']].drop_duplicates(subset='Passenger ID').copy()

In [48]:
dim_passenger.insert(0, 'passenger_key', range(1, len(dim_passenger) + 1))

In [49]:
dim_passenger.head(5)

,passenger_key,Passenger ID,Age,age_group,Gender,Nationality
0,1,ABVWIg,62,Middle Aged (45-64),Female,Japan
1,2,jkXXAX,62,Middle Aged (45-64),Male,Nicaragua
2,3,CdUz2g,67,Senior (65+),Male,Russia
3,4,BRS38V,71,Senior (65+),Female,China
4,5,9kvTLo,21,Young Adult (18-24),Male,China


In [50]:
dim_passenger = dim_passenger.rename(columns={
    'Passenger ID' : 'passenger_id',
    'Age'          : 'age',
    'Gender'       : 'gender',
    'Nationality'  : 'nationality'
})

dim_passenger = dim_passenger[['passenger_key', 'passenger_id', 'age', 'age_group', 'gender', 'nationality']]

In [51]:
dim_passenger.head(5)

,passenger_key,passenger_id,age,age_group,gender,nationality
0,1,ABVWIg,62,Middle Aged (45-64),Female,Japan
1,2,jkXXAX,62,Middle Aged (45-64),Male,Nicaragua
2,3,CdUz2g,67,Senior (65+),Male,Russia
3,4,BRS38V,71,Senior (65+),Female,China
4,5,9kvTLo,21,Young Adult (18-24),Male,China


# Creating dim_airport

In [54]:
dim_airport = pd.DataFrame()
print("Empty DataFrame created:")
display(dim_airport)

Empty DataFrame created:


""


In [55]:
#filter to only airports in the airline dataset
airline_codes = set(airline_df['Airport Code'].str.strip().str.upper())
dim_airport = airport_df[airport_df['iata_code'].str.upper().isin(airline_codes)].copy()

#select only the columns you need
dim_airport = dim_airport[['iata_code', 'name', 'type', 'municipality', 'iso_country']]

#map country_key from dim_location
loc_key_map = dict(zip(dim_location['country_code'], dim_location['location_key']))
dim_airport['country_key'] = dim_airport['iso_country'].map(loc_key_map)

#drop iso_country and rename
dim_airport = dim_airport.drop(columns='iso_country')
dim_airport.columns = ['airport_code', 'airport_name', 'airport_type', 'municipality', 'country_key']

#add airport_key
dim_airport.insert(0, 'airport_key', range(1, len(dim_airport) + 1))

In [56]:
dim_airport.head(5)

,airport_key,airport_code,airport_name,airport_type,municipality,country_key
0,1,THV,York Airport,small_airport,York,230
1,2,SLG,Smith Field,small_airport,Forney,230
2,3,UVA,Garner Field,small_airport,Brandywine,230
3,4,UTK,Utirik Airport,small_airport,Utirik Island,141
4,5,SVK,Silver Creek Airport,small_airport,Baker,230


# Creating dim_date

In [57]:
dim_date = pd.DataFrame()
print("Empty DataFrame created:")
display(dim_date)

Empty DataFrame created:


""


In [58]:
# Get unique dates and parse them
unique_dates = pd.to_datetime(airline_df['Departure Date'].dropna().unique(), format='%d/%m/%Y')
unique_dates = pd.Series(unique_dates).sort_values().reset_index(drop=True)

# Build dim_date
dim_date = pd.DataFrame({
    'date'    : unique_dates.dt.date,
    'day'     : unique_dates.dt.day,
    'week'    : unique_dates.dt.isocalendar().week.astype(int),
    'month'   : unique_dates.dt.month,
    'quarter' : unique_dates.dt.quarter,
    'year'    : unique_dates.dt.year,
})

dim_date.insert(0, 'date_key', range(1, len(dim_date) + 1))

In [59]:
dim_date.head(5)

,date_key,date,day,week,month,quarter,year
0,1,2022-01-01,1,52,1,1,2022
1,2,2022-01-02,2,52,1,1,2022
2,3,2022-01-03,3,1,1,1,2022
3,4,2022-01-04,4,1,1,1,2022
4,5,2022-01-05,5,1,1,1,2022


# Creating dim_flight_status

In [60]:
dim_flight_status = pd.DataFrame()
print("Empty DataFrame created:")
display(dim_flight_status)

Empty DataFrame created:


""


In [61]:
dim_flight_status = airline_df[['Flight Status']].drop_duplicates().copy()
dim_flight_status.columns = ['flight_status']
dim_flight_status.insert(0, 'flight_status_key', range(1, len(dim_flight_status) + 1))

In [62]:
dim_flight_status.head(5)

,flight_status_key,flight_status
0,1,On Time
3,2,Delayed
6,3,Cancelled


# Creating fact_flight_activity

In [63]:
fact_flight_activity = pd.DataFrame()
print("Empty DataFrame created:")
display(fact_flight_activity)

Empty DataFrame created:


""


In [64]:
#Handle missing airports before building the fact table

#Find airport codes in airline dataset that are missing from dim_airport
airline_codes = set(airline_df['Airport Code'].str.strip().str.upper())
airport_codes = set(dim_airport['airport_code'].str.strip().str.upper())
missing = airline_codes - airport_codes
print(f"Missing airport codes: {missing}")

#Build fallback rows from airline dataset for missing airports
fallback = (
    airline_df[airline_df['Airport Code'].str.upper().isin(missing)]
    [['Airport Code', 'Airport Name', 'Airport Country Code']]
    .drop_duplicates(subset='Airport Code')
    .copy()
)

#Map country_key and fill unknown columns
fallback['country_key']  = fallback['Airport Country Code'].str.upper().map(loc_key_map)
fallback['airport_type'] = None
fallback['municipality'] = None

#Rename to match dim_airport structure
fallback = fallback.rename(columns={
    'Airport Code': 'airport_code',
    'Airport Name': 'airport_name',
})
fallback = fallback[['airport_code', 'airport_name', 'airport_type', 'municipality', 'country_key']]

#Append fallback airports to dim_airport and reset surrogate keys
dim_airport = pd.concat([dim_airport, fallback], ignore_index=True)
dim_airport['airport_key'] = range(1, len(dim_airport) + 1)

Missing airport codes: {'MCQ', 'FAL', 'XMW', 'MSD', 'QOJ', 'KVD', 'AHT', 'ALV', 'QSA', 'ZCN', 'TXG', 'GML', 'KHL', 'PGP', 'QLF', 'LYM', 'QRT', 'MPK', 'BNJ', 'QUN', 'HOD', 'ALL', 'QNJ', 'SRI', 'CDF', 'FAJ', 'CWE', 'OUK', 'LUY', 'SII', 'SZT', 'GSY', 'QMH', 'CZW', 'MFT', 'MKA', 'DOK', 'CEJ', 'QGC', 'XDA', 'THF', 'HKV', 'CPQ', 'BXP', 'QMF', 'QLS', 'FDO', 'ZQL', 'UDN', 'QID', 'PSV', 'ZHI', 'ALY', 'ZRZ', 'BOF', 'JHQ', 'FEK', 'QXH', 'QNC', 'QYD', 'IOU', 'QSN', 'COX', 'SFR', 'TGV', 'XME', 'QGA', 'MSE', 'GDJ', 'JRS', 'EPS', 'AAG', 'TSE', 'YUJ', 'FEA', 'OSP', 'OSZ', 'LTR', 'VDB', 'QCN', 'QEO', 'KLJ', 'QPA', 'SZW', 'QKX', 'UNT', 'QNY', 'AYE', 'SMG', 'XQC', 'ECV', 'BNQ', 'RKO', 'UIQ', 'PHM', 'XAB', 'ZMG', 'TXL', 'FRU', 'XBX', 'DSA', 'JCA', 'RNR', 'QQT', 'QLA', 'QRC', 'QND', 'QHP', 'XCZ', 'XVS', 'KIV'}


In [65]:
#Build lookup maps from all dimension tables

passenger_map     = dict(zip(dim_passenger['passenger_id'],      dim_passenger['passenger_key']))
airport_map       = dict(zip(dim_airport['airport_code'],        dim_airport['airport_key']))
date_map          = dict(zip(dim_date['date'],                   dim_date['date_key']))
flight_status_map = dict(zip(dim_flight_status['flight_status'], dim_flight_status['flight_status_key']))

In [66]:
#Build the fact table using surrogate keys

fact_flight_activity['passenger_key']     = airline_df['Passenger ID'].map(passenger_map)
fact_flight_activity['airport_key']       = airline_df['Airport Code'].str.upper().map(airport_map)
fact_flight_activity['date_key']          = pd.to_datetime(airline_df['Departure Date'], format='%d/%m/%Y').dt.date.map(date_map)
fact_flight_activity['flight_status_key'] = airline_df['Flight Status'].map(flight_status_map)

#Add direct measure columns
fact_flight_activity['passenger_count']  = airline_df['passenger_count']
fact_flight_activity['is_international'] = airline_df['is_international']

#Add primary surrogate key
fact_flight_activity.insert(0, 'flight_activity_id', range(1, len(fact_flight_activity) + 1))


In [67]:
#Convert both to datetime.date format before mapping
date_map = dict(zip(
    pd.to_datetime(dim_date['date'], format='%Y-%m-%d').dt.date,
    dim_date['date_key']
))

fact_flight_activity['date_key'] = pd.to_datetime(
    airline_df['Departure Date'], format='%d/%m/%Y'
).dt.date.map(date_map)

#Verify
print(fact_flight_activity['date_key'].isnull().sum())  # should be 0

0


In [68]:
fact_flight_activity.head(5)

,flight_activity_id,passenger_key,airport_key,date_key,flight_status_key,passenger_count,is_international
0,1,1,5980,179,1,1,1
1,2,2,951,360,1,1,1
2,3,3,4627,18,1,1,1
3,4,4,1073,259,2,1,1
4,5,5,4252,56,1,1,1


In [69]:
display(fact_flight_activity.isnull().sum())

,0
flight_activity_id,0
passenger_key,0
airport_key,0
date_key,0
flight_status_key,0
passenger_count,0
is_international,0


# Exporting to csv

In [70]:
dim_location.to_csv('dim_location.csv', index=False)
dim_passenger.to_csv('dim_passenger.csv', index=False)
dim_airport.to_csv('dim_airport.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)
dim_flight_status.to_csv('dim_flight_status.csv', index=False)
fact_flight_activity.to_csv('fact_flight_activity.csv', index=False)